In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE

# Models
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
import xgboost as xgb



In [22]:
# Load dataset
df = pd.read_csv(r"C:\\Users\bhara\\OneDrive\\Desktop\\Bankruptcy Prediction\\data\\data.csv")

print("Dataset Shape:", df.shape)
print("First 5 rows:\n", df.head())

# Define features and target
X = df.drop("Bankrupt?", axis=1)
y = df["Bankrupt?"]

print("X and y created successfully!")

Dataset Shape: (6819, 96)
First 5 rows:
    Bankrupt?   ROA(C) before interest and depreciation before interest  \
0          1                                           0.370594          
1          1                                           0.464291          
2          1                                           0.426071          
3          1                                           0.399844          
4          1                                           0.465022          

    ROA(A) before interest and % after tax  \
0                                 0.424389   
1                                 0.538214   
2                                 0.499019   
3                                 0.451265   
4                                 0.538432   

    ROA(B) before interest and depreciation after tax  \
0                                           0.405750    
1                                           0.516730    
2                                           0.472295    
3        

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)


In [ ]:
# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# Define Models
# ==============================
models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": xgb.XGBClassifier(eval_metric='logloss'),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "Perceptron": Perceptron(),
    "AdaBoost": AdaBoostClassifier(),
    "Neural Network (MLP)": MLPClassifier(max_iter=500)
}

results = {}

In [ ]:
# Train & Evaluate
# ==============================
for name, model in models.items():
    print(f"\n===== {name} =====")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    
    roc = roc_auc_score(y_test, y_prob) if y_prob is not None else 0
    acc = model.score(X_test, y_test)
    
    results[name] = {"Accuracy": acc, "ROC-AUC": roc}
    


In [ ]:
# Convert to DataFrame
results_df = pd.DataFrame(results).T
print("\n=== Model Performance Summary ===")
print(results_df)


In [ ]:
# Mark Suitable / Not Suitable
results_df["Suitability"] = results_df.apply(
    lambda row: "✅ Suitable" if row["Accuracy"] > 0.85 and row["ROC-AUC"] > 0.80 else "❌ Not Suitable",
    axis=1
)
print("\n=== Suitability Check ===")
print(results_df)


In [ ]:
# ==============================
# Visualization
# ==============================

# --- Performance Comparison (Accuracy & ROC-AUC) ---
fig, ax = plt.subplots(figsize=(12,6))
results_df[["Accuracy", "ROC-AUC"]].plot(
    kind="bar", ax=ax, color=["#1abc9c", "#9b59b6"], edgecolor="black"
)
plt.title("Model Performance Comparison", fontsize=14, fontweight="bold")
plt.ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.ylim(0,1.05)

# Add value labels
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}", 
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=8, color="black", rotation=90)

plt.legend()
plt.show()

In [ ]:
# --- Suitability Graph ---
fig, ax = plt.subplots(figsize=(12,6))
colors = ["green" if s == "✅ Suitable" else "red" for s in results_df["Suitability"]]
bars = ax.bar(results_df.index, results_df["Accuracy"], color=colors, alpha=0.8, edgecolor="black")
plt.title("Model Suitability (Green = Suitable, Red = Not Suitable)", fontsize=14, fontweight="bold")
plt.ylabel("Accuracy")
plt.xticks(rotation=30, ha="right")

# Add labels ✅ ❌
for bar, label in zip(bars, results_df["Suitability"]):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01, label, ha='center', fontsize=9, fontweight="bold")

plt.ylim(0,1.05)
plt.show()

In [ ]:
# --- ROC Curves ---
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans'  # or 'Arial Unicode MS' if available

plt.figure(figsize=(10,7))
colors = [
    "blue","orange","green","red","purple","brown","pink","gray","cyan","magenta",
    "lime","gold","darkblue"
]
for (name, model), c in zip(models.items(), colors):
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, color=c, linewidth=2, label=name)

plt.plot([0,1],[0,1],'k--', linewidth=1)
plt.title("ROC Curves of Models", fontsize=14, fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore", message="X has feature names")


# Load your new data
file_path = ("C:\\Users\\bhara\\OneDrive\\Desktop\\Bankruptcy Prediction\\Data 2\\bankruptcy_features_95_nonan.csv")
new_data = pd.read_csv(file_path)
print("New data shape:", new_data.shape)
display(new_data.head())

# Validate columns
try:
    missing_cols = [col for col in X.columns if col not in new_data.columns]
    if missing_cols:
        print(f"\n⚠️ Missing columns: {missing_cols}")
    else:
        print("\n✅ All columns match training features.")
except NameError:
    print("\n⚠️ 'X' (training features) not found. Skipping column check.")

# Apply scaler if available
try:
    new_data_scaled = scaler.transform(new_data)
    X_new = new_data_scaled
    print("\n📏 Scaling applied.")
except Exception:
    print("\n(No scaler found — using raw data.)")
    X_new = new_data

# Predict
predictions = model.predict(X_new)
print("\n🔹 Predictions (first 10):")
print(predictions[:10])

# Probabilities (optional)
if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_new)
    print("\nPredicted probabilities (first 5 rows):")
    print(probs[:5])
